In [1]:
import json

%load_ext autoreload
%autoreload 2

from utgen.raggraph.walker import build_graph_from_directory
from utgen.raggraph.utils import get_node_context
from utgen.test_generation_crew.crew import TestGenerationCrew
from utgen.validation import validate_individual_test, save_and_clean_tests

In [2]:
g = build_graph_from_directory(code_path="../src", save_graph=False)

In [3]:
# Print nodes for inspection
for n, data in list(g.nodes(data=True))[10:20]:
    print(n, "\n\t", json.dumps(data, indent=2))
    print()

utgen/raggraph/parser.py::class::CodeGraphBuilder1 
	 {
  "type": "class",
  "name": "CodeGraphBuilder1",
  "file": "utgen/raggraph/parser.py",
  "signature": "CodeGraphBuilder1",
  "docstring": "First pass of code analysis: Builds nodes and structural edges.\n\nThis visitor creates nodes for files, classes, and functions/methods.\nIt establishes 'defines' and 'has_method' relationships.",
  "source": "class CodeGraphBuilder1(ast.NodeVisitor):\n    \"\"\"\n    First pass of code analysis: Builds nodes and structural edges.\n    \n    This visitor creates nodes for files, classes, and functions/methods.\n    It establishes 'defines' and 'has_method' relationships.\n    \"\"\"\n\n    def __init__(self, code_path: str, file_path: str, graph: nx.DiGraph):\n        \"\"\"\n        Initializes the first-pass builder.\n\n        Args:\n            code_path (str): The root directory of the project.\n            file_path (str): The specific file being visited.\n            graph (nx.DiGraph):

In [4]:
tests_results: dict[str, dict[str, dict[str, str]]] = {}

# TODO: afegir guardrails que falten
test_generator = TestGenerationCrew(guardrail_max_retries=3, verbose=False)

# Per a cada node del graf, obtenim el seu context i creem el test
for node_id, data in list(g.nodes(data=True))[10:20]:
    if data["type"] in ["function", "method"]:
        print(f"Generating tests for node: {node_id}")
        # Get context
        context = get_node_context(g=g, node_id=node_id)

        # Generate tests
        inputs = {"graph_context": context}
        response = test_generator.crew().kickoff(inputs=inputs)

        # Convert string to dictionary
        response_dict = json.loads(response.raw)

        # Store results
        k = data["file"].replace("/", "___").replace(".py", f"___{data['name']}")
        tests_results[k] = response_dict['tests']

Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder1.__init__
2026-03-16 10:38:35.369 | DEBUG    | utgen.test_generation_crew.guardrails:validate_tests_schema:40 - Guardrail input:
```json
{
  "tests": {
    "test_codegraphbuilder1_init_basic": {
      "name": "test_codegraphbuilder1_init_basic",
      "imports": [
        "import pytest",
        "from unittest.mock import patch, Mock",
        "import os",
        "import networkx as nx",
        "from utgen.raggraph.parser import CodeGraphBuilder1"
      ],
      "code": [
        "def test_codegraphbuilder1_init_basic():",
        "    # Arrange",
        "    code_path = \"/project/src\"",
        "    file_path = \"/project/src/module.py\"",
        "    graph = nx.DiGraph()",
        "",
        "    # Mock the canonical_id function to return a predictable value",
        "    with patch('utgen.raggraph.parser.canonical_id', return_value='module.py::file::module'):",
        "        # Act",
        "  


2026-03-16 10:38:35.372 | WARNING  | utgen.test_generation_crew.guardrails:validate_tests_schema:93 - Guardrail `validate_tests_schema` triggered: invalid test entries.


2026-03-16 10:38:35.374 | DEBUG    | utgen.test_generation_crew.guardrails:validate_tests_schema:95 - Guardrail `validate_tests_schema` validation failed:
- Test 'test_codegraphbuilder1_init_basic' schema error: 2 validation errors for TestCase
imports
  Input should be a valid string [type=string_type, input_value=['import pytest', 'from u...port CodeGraphBuilder1'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
code
  Input should be a valid string [type=string_type, input_value=['def test_codegraphbuild...node[\'source\'] == ""'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
- Test 'test_codegraphbuilder1_init_file_not_found' schema error: 2 validation errors for TestCase
imports
  Input should be a valid string [type=string_type, input_value=['import pytest', 'from u...port CodeGraphBuilder1'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v


2026-03-16 10:41:26.619 | WARNING  | utgen.test_generation_crew.guardrails:validate_tests_schema:93 - Guardrail `validate_tests_schema` triggered: invalid test entries.


2026-03-16 10:41:26.621 | DEBUG    | utgen.test_generation_crew.guardrails:validate_tests_schema:95 - Guardrail `validate_tests_schema` validation failed:
- Test 'test_visit_classdef_nested' syntax error: invalid syntax at line 28


KeyboardInterrupt: 

In [ ]:
for k1, v1 in tests_results.items():
    # Validem el test generat
    print(f"\nValidant tests per `{k1}`...")
    base_import = 'from ' + '.'.join(k1.split("___")[:-1]) + ' import ' + k1.split("___")[-1].split(".")[0]
    print(f"Base import afegit: {base_import}")
    filename = f"test_{k1.replace('.', '_')}.py"
    tests_que_han_passat = []

    for k2, v2 in v1.items():
        name, imp, code = k2, v2['imports'], v2['code']
        # Afegim el import base per si no està present
        if base_import not in imp:
            print(f"⚠️ Import base `{base_import}` no present en el test `{name}`, afegint-lo...")
            imp.append(base_import)
        imp = "\n".join(imp)
        if validate_individual_test(import_code=imp, test_code=code):
            print(f"✅ Test `{name}` acceptat")
            tests_que_han_passat.append((imp, code))
        else:
            print(f"❌ Test `{name}` rebutjat")
        print(imp)
        print(code)

    # Guardem i deixem el fitxer perfecte
    save_and_clean_tests(valid_tests=tests_que_han_passat, destination=f"../tests/{filename}")

# TODO: script final que reordene la carpeta tests: un test.py per cada .py del src
# TODO: run pytest coverage


Validant tests per `utgen___validation___guardar_i_netejar_tests`...
Base import afegit: from utgen.validation import guardar_i_netejar_tests
❌ Test `test_guardar_i_netejar_tests_empty_list` rebutjat
import os
from unittest.mock import patch
from utgen.validation import guardar_i_netejar_tests
from utgen.validation import guardar_i_netejar_tests
@patch('builtins.print')
@patch('os.makedirs')
@patch('subprocess.run')
def test_guardar_i_netejar_tests_empty_list(mock_subprocess, mock_makedirs, mock_print):
    guardar_i_netejar_tests([])
    mock_print.assert_called_once_with("No hi ha tests vàlids per guardar.")
    mock_makedirs.assert_not_called()
    mock_subprocess.assert_not_called()
❌ Test `test_guardar_i_netejar_tests_basic` rebutjat
import os
from unittest.mock import patch, mock_open
from utgen.validation import guardar_i_netejar_tests
from utgen.validation import guardar_i_netejar_tests
@patch('builtins.print')
@patch('os.makedirs')
@patch('subprocess.run')
@patch('builtins.op

In [7]:
import subprocess

full_code = """import os
from unittest.mock import patch
from utgen.validation import guardar_i_netejar_tests
from utgen.validation import guardar_i_netejar_tests
@patch('builtins.print')
@patch('os.makedirs')
@patch('subprocess.run')
def test_guardar_i_netejar_tests_empty_list(mock_subprocess, mock_makedirs, mock_print):
    guardar_i_netejar_tests([])
    mock_print.assert_called_once_with("No hi ha tests vàlids per guardar.")
    mock_makedirs.assert_not_called()
    mock_subprocess.assert_not_called()"""

temp_filename = "temp_validation.py"

with open(temp_filename, "w") as f:
    f.write(full_code)

# Executem pytest. -q per silenci, --tb=no per no embrutar la consola
result = subprocess.run(["pytest", temp_filename, "-q", "--tb=no"], capture_output=True)
